# 04 · MLSD (distância do disparo) no mesmo protocolo — cabeças e baselines

**Sessão P1 · ambiente L4.** Mesma receita, mesmas dobras e mesmo código do 02/03, agora na tarefa de **distância** (3 classes: encostado, curta, distância), **só nas entradas** — como a MLSD de Lira et al. (2025, IJLM) e o `TransferKAN_MLSD` antigo (que fica intacto como registro histórico).

* **Dobras:** as de `folds_entrada_saida.csv` (agrupadas por caso, estratificadas por `strat4`). Na unidade da dobra *k*, a coluna `fold_dist_dk` põe no treino só as **entradas** das outras dobras; o teste é a dobra *k* inteira — as saídas só recebem predição (para a métrica de pipeline do P2) e **não entram nas métricas**, que ignoram `y_dist = −1`.
* **A nova TransferKAN-MLSD é a cabeça (d):** ResNet-152 ImageNet → Dropout 0,5 → KAN `[2048,128,3]`, grade 10, ordem 3; 2 fases (15 + 45 épocas), última época, CE com pesos `n/(K·n_c)`, augmentation "como rodou" — idêntica à de entrada/saída.

| id | cabeça | parâmetros da cabeça |
|---|---|---|
| (a) | `Linear(2048,3)` — também é a linha "ResNet152" dos baselines | 6.147 |
| (b) | MLP `[2048,128,3]` | 262.659 |
| (c) | MLP isoparamétrica `[2048,1919,3]` | 3.937.791 |
| (d) | **KAN `[2048,128,3]` = TransferKAN-MLSD** | 3.937.920 |
| (e) | KAN `[2048,8,3]` (opcional) | 246.120 |

**Etapas** (`ETAPA`):
* `"cabecas_s42"` = (d) → (c) → (b) → (a), semente 42 (**já responde "a KAN melhora o MLSD?"**, ~2,1 h);
* `"cabecas_s43_44"` = as mesmas com sementes 43 e 44 (~4,1 h; duas abas: `FILTRO_SEMENTES = [43]` numa e `[44]` na outra);
* `"baselines"` = AlexNet → MobileNetV2 → MNASNet1.3 → ShuffleNetV2 x0.5 → SqueezeNet1.0 → DenseNet169 → MaxViT-T → Swin-B, semente 42 (~3,3 h);
* `"pesados"` = EfficientNetV2-L → VGG16 → RegNetY-16GF, semente 42 (~2,3 h);
* `"baselines_sementes_extras"` = Swin-B e MaxViT-T com sementes 43 e 44 (~3,4 h; checagem de robustez, como em entrada/saída);
* `"e"` = cabeça (e), opcional.

**Como rodar:** igual ao 02 (`TESTE_RAPIDO = True` na 1ª vez; depois `False` e **Executar tudo**; se cair, **Executar tudo** de novo — unidades prontas são puladas e a do meio retoma da última época salva).

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o dataset é clonado e o cache é gerado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_PLUGIN_ESPERADO = "e2138640a83ef7587200ad6e051262b6a49e5b60027f6b4a61407fd1f61889f7"   # configs_mlsd.py
ABA = "A"                    # dono das travas desta aba (2ª aba simultânea: "B")
PERMITIR_RESERVA = False     # True só para rodar na T4 como reserva (marcado com † na tabela)
ETAPA = "cabecas_s42"        # "cabecas_s42" | "cabecas_s43_44" | "baselines" | "pesados" | "baselines_sementes_extras" | "e"
SO_MODELOS = None            # ex.: ["swin_b"] (só nas etapas de baselines)
FILTRO_SEMENTES = None       # ex.: [43] numa aba e [44] na outra
TESTE_RAPIDO = True          # 1ª vez True; depois False
CKPT_INTERVALO_MIN = None    # None = usa a recomendação do 01 (0 = checkpoint a cada época)

In [ ]:
# Dependências fixadas (rode ANTES de importar qualquer coisa da albumentations)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
ARQ_PLUGIN = f"{DIR_DRIVE}/configs_mlsd.py"
assert pc.sha256_arquivo(ARQ_PLUGIN) == SHA_PLUGIN_ESPERADO, (
    "O configs_mlsd.py do Drive não é o publicado. Não rode nada; avise o P1.")
pc.carregar_plugin(ARQ_PLUGIN)          # registra as configs mlsd__* (o SHA vai para cada metricas.json)
cm = sys.modules["configs_mlsd"]
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "… | plugin", SHA_PLUGIN_ESPERADO[:16], "…")
GPU = pc.exigir_gpu(permitir_reserva=PERMITIR_RESERVA)   # rodadas comparáveis só na L4
pc.definir_id_sessao(ABA)
print("GPU:", GPU["nome"], "| comparável:", GPU["comparavel"], "| dono das travas:", pc.ID_SESSAO)

In [ ]:
# Clona o dataset e o leguwoi nos commits fixados e gera o cache 224×224 só nesta sessão (~3–5 min)
sess = pc.preparar_sessao(
    RAIZ_TRABALHO,
    caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}",
    caminho_meta=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_META_DOBRAS}",
)
R, C = f"{DIR_DRIVE}/resultados", f"{DIR_DRIVE}/checkpoints"
RT, CT = f"{DIR_DRIVE}/resultados_teste", f"{DIR_DRIVE}/checkpoints_teste"
print(json.dumps(sess.meta, indent=1))
if sess.meta.get("cache_confere") is False:
    print("⚠️ O cache diverge do registrado no 00 (versão do OpenCV?). Divergência:", sess.meta.get("cache_divergencia"))
sess.df = cm.adicionar_colunas_dist(sess.df)       # fold_dist_d0..d4
sess.meta["mlsd"] = "treino só nas entradas; teste = dobra inteira (saídas sem métrica)"
print("Dobras da MLSD (teste da dobra k; as saídas nunca treinam):")
display(cm.checar_colunas_dist(sess.df))

In [ ]:
cal = json.loads((Path(R) / "calibracao_l4.json").read_text())
PREC = cal.get("decisao_precisao")
assert PREC in ("amp", "fp32"), "Rode o 01 (ponte fp32 × AMP) antes."
CKPT = cal.get("recomendacao_ckpt_intervalo_min", 0.0) if CKPT_INTERVALO_MIN is None else CKPT_INTERVALO_MIN
H, B = cm.CONFIGS_CABECA, cm.CONFIGS_BASELINES
if ETAPA in ("cabecas_s42", "cabecas_s43_44"):
    exp, configs = "mlsd_cabeca", [H["d"], H["c"], H["b"], H["a"]]
    sementes = [42] if ETAPA == "cabecas_s42" else [43, 44]
elif ETAPA == "e":
    exp, configs, sementes = "mlsd_cabeca", [H["e"]], [42]
elif ETAPA in ("baselines", "pesados", "baselines_sementes_extras"):
    grupo = {"baselines": pc.BASELINES[:8], "pesados": pc.BASELINES[8:],
             "baselines_sementes_extras": ["swin_b", "maxvit_t"]}[ETAPA]
    modelos = [b for b in grupo if SO_MODELOS is None or b in SO_MODELOS]
    exp, configs = "mlsd_baselines", [B[b] for b in modelos]
    sementes = [43, 44] if ETAPA == "baselines_sementes_extras" else [42]
else:
    raise ValueError(ETAPA)
if FILTRO_SEMENTES:
    sementes = [s for s in sementes if s in FILTRO_SEMENTES]
fila = cm.montar_fila_dist(exp, configs, sementes=sementes)
print(f"Experimento {exp} | precisão: {PREC} | checkpoint a cada ≥ {CKPT} min | {len(fila)} unidades:")
print(pd.DataFrame(fila).groupby(["config", "semente"]).size())

## Teste rápido (1 dobra × 2 épocas do 1º e do último modelo da etapa, pasta de teste)

In [ ]:
if TESTE_RAPIDO:
    import shutil
    for u0 in dict.fromkeys([fila[0]["config"], fila[-1]["config"]]):
        u = {**[x for x in fila if x["config"] == u0][0], "experimento": "t04_rapido", "dobra": 0,
             "coluna_dobra": cm.coluna_dobra_dist(0)}
        shutil.rmtree(f"{RT}/t04_rapido/{u['config']}", ignore_errors=True)
        shutil.rmtree(f"{CT}/t04_rapido/{u['config']}", ignore_errors=True)
        r = pc.executar_unidade(sess, u, RT, CT, precisao=PREC, limite_epocas=(1, 1), espera_trava_s=0)
        m = r["metricas"]["dist"]
        print(u["config"], "OK:", {k: round(m[k], 4) for k in ("acc", "auc", "f1_macro")},
              f"| teste {r['n_teste']} imagens ({m['n']} entradas na métrica) | treino {r['n_treino']} | tempos", r["tempos"])
    print("➡️ Agora mude TESTE_RAPIDO = False e rode tudo de novo.")

## Rodada

In [ ]:
assert not TESTE_RAPIDO, "TESTE_RAPIDO está True: mude para False para rodar a etapa."
feitos = pc.executar_fila(sess, fila, R, C, precisao=PREC, ckpt_intervalo_min=CKPT)
print("Concluídas nesta sessão:", len(feitos))

## Resultados (runs.csv, resumo.json, oof.csv de cada experimento)

In [ ]:
METR = ["dist_acc", "dist_prec_macro", "dist_rec_macro", "dist_f1_macro", "dist_acc_balanceada", "dist_auc"]

def agregar_experimento(experimento):
    """Grava runs.csv (1 linha = modelo × semente × dobra), resumo.json e oof.csv do experimento."""
    runs = pc.ler_resultados(R, experimento)
    if runs.empty:
        print("Nada concluído ainda em", experimento)
        return runs
    runs = runs[~runs["teste_rapido"].astype(bool)].copy()
    d = Path(R) / experimento
    runs.to_csv(d / "runs.csv", index=False)
    resumo = {}
    for (cfg, sem), g in runs.groupby(["config", "semente"]):
        resumo.setdefault(cfg, {})[f"s{sem}"] = {
            "n_dobras": int(len(g)),
            **{m: {"media": float(g[m].mean()), "dp": float(g[m].std(ddof=1)) if len(g) > 1 else None} for m in METR},
            "treino_min_por_dobra": float(g["treino_min"].mean()),
            "params_cabeca": int(g["params_cabeca"].iloc[0]),
            "params_total": int(g["params_total"].iloc[0]),
            "gpus": sorted(g["gpu"].dropna().unique().tolist()),
        }
    pc.gravar_json_atomico(resumo, d / "resumo.json")
    partes = []
    for _, r in runs.iterrows():
        p = pd.read_csv(Path(r["caminho"]) / "pred_teste.csv")
        partes.append(p.assign(config=r["config"], semente=r["semente"], dobra=r["dobra"]))
    pd.concat(partes).to_csv(d / "oof.csv", index=False)
    print(f"{experimento}: {len(runs)} unidades -> runs.csv, resumo.json, oof.csv")
    return runs

for experimento in ("mlsd_cabeca", "mlsd_baselines"):
    runs = agregar_experimento(experimento)
    if not runs.empty:
        display(runs.groupby(["config", "semente"])[["dist_acc", "dist_f1_macro", "dist_acc_balanceada", "dist_auc", "treino_min"]]
                .agg(["mean", "std", "count"]).round(4))

⚠️ Ao terminar, desconecte o ambiente (*Ambiente de execução → Desconectar e excluir*) para não consumir unidades parado.